# Stage 3: TLS regions

Thirteen regions, each cropped from the raw Xenium output: 8 TLS in PDAC, one lymph node region in MH104 used as the secondary lymphoid structure (SLS) reference, and 4 TLS in TNBC. Each region gets its own QC and normalisation, then cells are labelled by gene-set scoring within the region (`TLS_cell_class_threshold`), and neighbourhood enrichment is computed on those labels.

The steps follow the original notebooks, one per region. PCA, UMAP, Leiden clustering and Moran's I are left out, because the labels and the figures don't depend on them.

**TLS_097_480_n2:** the FDCs gene set is `['VCAM1','FDCSP','CR1','CR2']`, the set that made the original labels. The saved notebook ran the labelling cell before its gene-set cell was changed to the current FDCs set, so the current set reproduces only 98.44% of labels; the earlier set reproduces 100% of labels and of the saved `max_score`.

**SLS_ref:** "Unclassified" cells are labelled "B cells", as in the original object. This step isn't in the saved notebook; it was found by comparing labels: every SLS_ref label matched except the 1,281 cells below the threshold, all of which were "B cells" in the original, and all scores were identical.

Three misspelt genes in the original gene sets are removed: `RSG5` (SMCs, 314 samples), `CITTA` and `IFR8` (pDCs, TNBC). They are not in the panels, so scanpy ignored them in the original runs as well and the scores are unchanged.

**TRCs:** the label called `Stromal_CCL19` in the original notebooks is called `TRCs` here, to match the rest of the analysis. Only the name changes: the gene sets and the cells are the same. The check at the end maps the original name to `TRCs` before comparing.

**Saved outputs:** this notebook was run before `Stromal_CCL19` was renamed `TRCs`. The text outputs have been updated to `TRCs`, but the saved plot images still show `Stromal_CCL19`, which is the same label as `TRCs`. The check output comes from the code before the rename, which compared the labels directly.

## Parameters

In [ ]:
# Folders and input files: set these before running (papermill parameters)
# Raw Xenium output, one subfolder per sample (cell_feature_matrix.h5 and cells.csv)
RAW_DATA_DIR = "/path/to/raw_xenium"
# Raw Xenium output of the four TNBC sections (GEO GSE281886), one subfolder per section
TNBC_RAW_DIR = "/path/to/raw_xenium_tnbc"
OUTPUT_DIR = "/path/to/stage3_output/TLS_regions"
# The original region objects, used only for the check at the end; set a path to None to skip it
ARCHIVED_DIR = "/path/to/original_regions"

In [1]:
import os

RAW_DIRS = {
    "TLS_107_314": f"{RAW_DATA_DIR}/MH107_314",
    "TLS_104_314": f"{RAW_DATA_DIR}/MH104_314",
    "TLS_065_314": f"{RAW_DATA_DIR}/MH065_314",
    "TLS_096_480_n2": f"{RAW_DATA_DIR}/MH096_480",
    "TLS_096_480_n1": f"{RAW_DATA_DIR}/MH096_480",
    "TLS_097_480_n3": f"{RAW_DATA_DIR}/MH097_480",
    "TLS_097_480_n2": f"{RAW_DATA_DIR}/MH097_480",
    "TLS_097_480_n1": f"{RAW_DATA_DIR}/MH097_480",
    "SLS_ref": f"{RAW_DATA_DIR}/MH104_480",
    "TNBC_1": f"{TNBC_RAW_DIR}/TNBC_1",
    "TNBC_2": f"{TNBC_RAW_DIR}/TNBC_2",
    "TNBC_3": f"{TNBC_RAW_DIR}/TNBC_3",
    "TNBC_4": f"{TNBC_RAW_DIR}/TNBC_4",
}
ARCHIVED_FILES = {
    "TLS_107_314": f"{ARCHIVED_DIR}/MH107_TLS_314_ss_cropped_#1.h5ad",
    "TLS_104_314": f"{ARCHIVED_DIR}/MH104_TLS_314_ss_cropped_#1.h5ad",
    "TLS_065_314": f"{ARCHIVED_DIR}/MH065_TLS_314_ss_cropped_#1.h5ad",
    "TLS_096_480_n2": f"{ARCHIVED_DIR}/MH096_TLS_480_ss_cropped_#2.h5ad",
    "TLS_096_480_n1": f"{ARCHIVED_DIR}/MH096_TLS_480_ss_cropped_#1.h5ad",
    "TLS_097_480_n3": f"{ARCHIVED_DIR}/MH097_TLS_480_ss_cropped_#3.h5ad",
    "TLS_097_480_n2": f"{ARCHIVED_DIR}/MH097_TLS_480_ss_cropped_#2.h5ad",
    "TLS_097_480_n1": f"{ARCHIVED_DIR}/MH097_TLS_480_ss_cropped_#1.h5ad",
    "SLS_ref": f"{ARCHIVED_DIR}/MH104_SLS_480plex_setseed.h5ad",
    "TNBC_1": f"{ARCHIVED_DIR}/Xenium_BreastTLS_0010466top_cropped.h5ad",
    "TNBC_2": f"{ARCHIVED_DIR}/BreastTLS_0010466b_cropped.h5ad",
    "TNBC_3": f"{ARCHIVED_DIR}/BreastTLS_0010544top_cropped.h5ad",
    "TNBC_4": f"{ARCHIVED_DIR}/BreastTLS_0010544b_cropped.h5ad",
}
# Labels renamed from the original notebooks, for the check against the original objects
RENAMED_LABELS = {"Stromal_CCL19": "TRCs"}

# Region as (x_min, x_max, y_min, y_max) in µm, minimum counts per cell, score quantile for labelling,
# the neighbourhood enrichment settings (radius None = squidpy's default 6 nearest neighbours),
# and for SLS_ref the label given to cells below the threshold
tls_regions = {
    "TLS_107_314": {"box": (7000, 8500, 1000, 4400), "min_counts": 15, "quantile": 0.65, "radius": None, "seed": 1122},
    "TLS_104_314": {"box": (2500, 4400, 700, 2000), "min_counts": 12, "quantile": 0.75, "radius": None, "seed": 1122},
    "TLS_065_314": {"box": (2500, 4000, 3200, 4200), "min_counts": 20, "quantile": 0.80, "radius": None, "seed": 1122},
    "TLS_096_480_n2": {"box": (600, 2500, 3170, 4255), "min_counts": 10, "quantile": 0.80, "radius": None, "seed": 1122},
    "TLS_096_480_n1": {"box": (3500, 5100, 1700, 3500), "min_counts": 15, "quantile": 0.75, "radius": None, "seed": 1122},
    "TLS_097_480_n3": {"box": (4400, 6500, 1700, 3500), "min_counts": 15, "quantile": 0.75, "radius": None, "seed": 1122},
    "TLS_097_480_n2": {"box": (4000, 6500, 300, 1600), "min_counts": 15, "quantile": 0.75, "radius": None, "seed": 1122},
    "TLS_097_480_n1": {"box": (880, 2800, 50, 1900), "min_counts": 10, "quantile": 0.75, "radius": None, "seed": 1122},
    "SLS_ref": {"box": (7000, 8700, 190, 1900), "min_counts": 20, "quantile": 0.70, "radius": 50, "seed": 1122,
                "unclassified_to": "B cells"},
    "TNBC_1": {"box": (1500, 5000, 1000, 3000), "min_counts": 15, "quantile": 0.75, "radius": None, "seed": 11223},
    "TNBC_2": {"box": (1, 2300, 3900, 9000), "min_counts": 10, "quantile": 0.70, "radius": None, "seed": 11223},
    "TNBC_3": {"box": (4000, 6500, 1000, 2800), "min_counts": 10, "quantile": 0.75, "radius": None, "seed": 11223},
    "TNBC_4": {"box": (5300, 8500, 4000, 6000), "min_counts": 15, "quantile": 0.75, "radius": None, "seed": 11223},
}

In [2]:
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq

os.makedirs(OUTPUT_DIR, exist_ok=True)

~/miniconda3/envs/xenium.squidpy/lib/python3.9/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (
~/miniconda3/envs/xenium.squidpy/lib/python3.9/site-packages/dask/dataframe/_pyarrow_compat.py:15: FutureWarning: Minimal version of pyarrow will soon be increased to 14.0.1. You are using 13.0.0. Please consider upgrading.
  warnings.warn(
~/miniconda3/envs/xenium.squidpy/lib/python3.9/site-packages/numba/core/decorators.py:246: RuntimeWarning: nopython is set for njit and is ignored
  warnings.warn('nopython is set for njit and is ignored', RuntimeWarning)


## Gene sets for labelling, from each region's original notebook

In [3]:
gene_sets_by_region = {}
# 'TRCs' was called 'Stromal_CCL19' in the original notebooks

gene_sets_by_region["TLS_107_314"] = {
    'Malignant cells': ['KRT19','KRT7','KRT8','MUC1'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Ductal MUC6': ['MUC6','REG1A','CFTR','SLC4A4'],
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    #'Endocrine': ['PPY','INS','GCG','SST'],
    # TLS 
    #'FDCs': ['VCAM1','FDCSP','CR2','CCL19'],
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','IGF1','CXCL14'], 
    'TRCs' : ['PDGFRA','CCL19'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['PDGFRB','ACTA2','CALD1'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    'Schwann': ['CDH19','CHL1'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['ADIPOQ','PLIN1','LIPE'],
    # CAFs
    'CAFs_MYO': ['COL11A1','POSTN','ACTA2','PDGFC'],
    'CAFs_CXCL14': ['COL11A1','CXCL14'], # Not many CXCL14, mostly CAFs-MYO
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    #'CD4 Tfh' : ['CD4','ICOS','PDCD1'], #very very few
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CXCL13','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1'],
    #'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    #'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    #'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['FPR3','LIPA','SPP1','MARCO'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','C1QB','HLA-DMB'],
    'TAMs_MHC' : ['CD86','HLA-DRB1','HLA-DRB5','CSF1R','CLEC5A']
}

gene_sets_by_region["TLS_104_314"] = {
    'Malignant cells': ['KRT19','KRT7','KRT8','MUC1'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Ductal MUC6': ['MUC6','REG1A','CFTR','SLC4A4'],
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    #'Endocrine': ['PPY','INS','GCG','SST'],
    # TLS 
    #'FDCs': ['VCAM1','FDCSP','CR2','CCL19'],
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','IGF1','CXCL14'], 
    'TRCs' : ['PDGFRA','CCL19'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['PDGFRB','ACTA2','CALD1'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    'Schwann': ['CDH19','CHL1'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['ADIPOQ','PLIN1','LIPE'],
    # CAFs
    'CAFs_MYO': ['COL11A1','POSTN','ACTA2','PDGFC'],
    'CAFs_CXCL14': ['COL11A1','CXCL14'], # Not many CXCL14, mostly CAFs-MYO
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    #'CD4 Tfh' : ['CD4','ICOS','PDCD1'], #very very few
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CXCL13','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1'],
    #'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    #'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    #'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['FPR3','LIPA','SPP1','MARCO'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','C1QB','HLA-DMB'],
    'TAMs_MHC' : ['CD86','HLA-DRB1','HLA-DRB5','CSF1R','CLEC5A']
}

gene_sets_by_region["TLS_065_314"] = {
    'Malignant cells': ['KRT19','KRT7','KRT5','KRT17'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    'Ductal MUC6': ['MUC6','REG1A','CFTR','SLC4A4'],
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    'Endocrine': ['PPY','INS','GCG','SST'],
    # TLS 
    #'FDCs': ['VCAM1','FDCSP','CR2','CCL19'],
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','IGF1','CXCL14'], 
    'TRCs' : ['PDGFRA','CCL19'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['PDGFRB','ACTA2','CALD1'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    #'Schwann': ['CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['ADIPOQ','PLIN1','LIPE'],
    # CAFs
    'CAFs_MYO': ['FN1','MMP11','COL11A1','ACTA2'],
    'CAFs_CXCL14': ['FN1','MMP11','COL11A1','CXCL14'], # Not many CXCL14, mostly CAFs-MYO
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CXCL13','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1'],
    #'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    #'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    #'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    #'TAMs_SPP1' : ['LIPA','SPP1','MARCO','CHIT1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','C1QB','HLA-DMB'],
    'TAMs_MHC' : ['CD86','HLA-DRB1','HLA-DRB5','CSF1R','CLEC5A']
}

gene_sets_by_region["TLS_096_480_n2"] = {
    'Malignant cells': ['KRT8','KRT19','KRT7','KRT17'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    #'Endocrine': ['PPY','INS'],
    # TLS 
    'FDCs': ['VCAM1','FDCSP','CR2','CCL19'],
    'FRCs': ['PLA2G2A','IL1R1','PDPN','PDGFRA','MEDAG'], 
    'TRCs' : ['PDGFRA','CCL19','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['MYH11','PDGFRB'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    'Adipocytes': ['ADIPOQ','PLIN1','LIPE'],
    # CAFs
    'CAFs_MYO': ['POSTN','MMP11','COL11A1'],
    #'CAFs_CXCL14': ['COL11A1','CXCL14','IGFBP5','POSTN'], # Not many CXCL14, mostly CAFs-MYO
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['LIPA','SPP1','MARCO','CHIT1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["TLS_096_480_n1"] = {
    'Malignant cells': ['KRT19', 'S100A6','EPCAM','KRT8','KRT17','KRT7'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    #'Endocrine': ['PPY','INS'],
    # TLS 
    'FDCs': ['VCAM1','FDCSP','CR1','CR2'],
    'FRCs': ['PLA2G2A','ICAM1','PDPN','PDGFRA','MEDAG'], #very few
    'TRCs' : ['PDGFRA','CCL19','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['MYH11','PDGFRB'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    'Adipocytes': ['ADIPOQ','PLIN1','LIPE'],
    # CAFs
    'CAFs_MYO': ['POSTN','MMP11','COL11A1'],
    'CAFs_CXCL14': ['COL11A1','CXCL14','IGFBP5','POSTN'],
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['LIPA','SPP1','MARCO','CHIT1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["TLS_097_480_n3"] = {
    'Malignant cells': ['KRT17','KRT19','KRT7','KRT8'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    'Ductal PanIN': ['MUC6','REG1A','CFTR','MUC3A'],
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    'Endocrine': ['PPY','INS'],
    # TLS 
    'FDCs': ['VCAM1','SLC1A2','RGS9','FDCSP','CR1','CR2'],
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','MEDAG'], #very few
    'TRCs' : ['PDGFRA','CCL19','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5','CSPG4'],
    'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'CAFs_MYO': ['POSTN','MMP11','PDGFC'],
    'CAFs_CXCL14': ['POSTN','CXCL14','IGFBP3','WNT5A'],
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1','HLA-DRA'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['LIPA','SPP1','MARCO','CHIT1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["TLS_097_480_n2"] = {
    'Malignant cells': ['KRT19','KRT17','SLC2A1','KRT7'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    'Ductal PanIN': ['MUC6','REG1A','CFTR','MUC3A'],
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    'Endocrine': ['PPY','INS'],
    # TLS 
    'FDCs': ['VCAM1','SLC1A2','RGS9','FDCSP','CR1','CR2'],
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','MEDAG'], #very few
    'TRCs' : ['PDGFRA','CCL19','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5','CSPG4'],
    'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'CAFs_MYO': ['POSTN','MMP11','PDGFC'],
    'CAFs_CXCL14': ['POSTN','CXCL14','IGFBP3'],
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1','HLA-DRA'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    #'TAMs_SPP1' : ['SPP1','LIPA','CD163','MARCO','CHIT1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["TLS_097_480_n1"] = {
    #'Malignant cells': ['KRT19', 'S100A6','EPCAM','MUC1','KRT8','KRT7'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    'Endocrine': ['PPY','INS'],
    # TLS 
    'FDCs': ['VCAM1','FDCSP','CR1','CR2'],
    'FRCs': ['PLA2G2A','ICAM1','PDPN','PDGFRA','MEDAG'], #very few
    'TRCs' : ['PDGFRA','CCL19','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    'SMCs' : ['MYH11','PDGFRB'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5','CSPG4'],
    'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'CAFs_MYO': ['POSTN','MMP19'],
    #'CAFs_CXCL14': ['COL11A1','CXCL14','IGFBP5'],
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','FCGR3B','TREM1'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['LIPA','CD163','CSTB','SLC2A1'], # 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["SLS_ref"] = {
    'Malignant cells': ['KRT19', 'S100A6','EPCAM','MUC1','KRT8','KRT7'],
    #'IPMN': {'MUC2','REG4','SPINK4'}, 
    #'Ductal MUC5B': {'MUC6','TFF2','TFF3','FCGBP','CSF1R','SLC4A4'},
    #'Ductal' : {'CFTR','SLC4A4','PKHD1','CRP'},
    #'Acinar' : {'CTRC','CUZD1','PRSS1','AMY2A'},
    #'Acinar_REG+' : {'REG1B','REG3G','REG3A'},
    #'Malignant acinar' : {'MUCL3','NPSR1','PCDH7','REG1B','PRSS1'},
    #'Tuft' : {'AVIL','TRPM5','BMX'},
    #'Endocrine': {'INS','GCG','SST'},
    # TLS 
    'FDCs': ['VCAM1','SLC1A2','RGS9'],
    'FRCs': ['PLA2G2A','ICAM1','PDPN','PDGFRA'], #very few
    'TRCs' : ['PDGFRA','CCL19','CCL21','COL6A3','VCAM1'], #,'CXCL12',
    #'HEV': ['MEOX2'], #,'CHST4','ANKRD53'
    #'LECs': ['LYVE1','FLT4','PROX1'], #NO LEC inside 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR','MEOX2'],
    'SMCs' : ['MYH11','PDGFRB'], #don't use ACTA2
    #'Pericytes': ['PDGFRB','TRPC6','CSPG4'],
    #'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': {'SYT1','ADGRB3','KCNIP4','CHAT','TAC1'},
    'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'CAFs_MYO': ['COL11A1','POSTN','MMP11'],
    #'CAFs_CXCL14': ['COL11A1','CXCL14','IGFBP5'],
    'CAFs_NRT' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','KCNIP1','ABCA8'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD19','FCRL1'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','JCHAIN'],
    #'gdT' : ['XCL1','TRGC1','TRGC2'],
    # Myeloid
    'Neutrophils': ['CSF3R','TREM1','FCGR3B'],
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['LILRA4','IL3RA'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'TAMs_SPP1' : ['LIPA','MARCO','HK2','CSTB','SLC2A1'], # 'SPP1' expressed by endothelial, ductal and TAMs
    'TAMs_OLR1' : ['OLR1','HBEGF'],
    'TAMs_C1QC' : ['C1QC','SELENOP','C1QA','F13A1','SLC40A1'],
    'TAMs_MHC' : ['CD86','IL1R2','HLA-DRB1','HLA-DRB5','CSF1R']
}

gene_sets_by_region["TNBC_1"] = {
    'Malignant cells': ['EPCAM','KRT7','KRT8','NECTIN4'],
    'Myoepithelial':['KRT5','ACTA2'],
    # TLS 
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','TNFSF14'], #very few
    'TRCs' : ['PDGFRA','CCL19','ICAM1'], #,'CXCL12',
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    #'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    #'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'myCAFs': ['POSTN','MMP2','COL11A1'],
    'iCAFs' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7','CCR7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1','CXCL13'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD52','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','PRDM1'],
    # Myeloid
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['GZMB'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    'LAM' : ['FPR3','APOE','APOC1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'MAC_SIGLEC1' : ['SELENOP','CSF1R','SIGLEC1','CD163']
}

gene_sets_by_region["TNBC_2"] = {
    #'Malignant cells': ['EPCAM','KRT7','KRT8','NECTIN4'],
    #'Myoepithelial':['KRT5','ACTA2'],
    # TLS 
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','TNFSF14'], #very few
    #'FDCs': ['ICAM1','PDGFRA','CXCL13'], # check if exist
    'TRCs' : ['PDGFRA','CCL19','ICAM1'], #,'CXCL12',
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    #'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    #'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'myCAFs': ['POSTN','MMP2','FN1'],
    'iCAFs' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7','CCR7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    #'CD4 Tfh' : ['CD4','ICOS','PDCD1','CXCL13'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD52','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','PRDM1'],
    # Myeloid
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['GZMB'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','KIT','TPSAB1'], 
    #'LAM' : ['FPR3','APOE','APOC1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'MAC_SIGLEC1' : ['SELENOP','CSF1R','SIGLEC1','CD163']
}

gene_sets_by_region["TNBC_3"] = {
    'Malignant cells': ['EPCAM','KRT7','KRT8','NECTIN4'],
    'Myoepithelial':['KRT5','ACTA2'],
    'Luminal progenitor':['KRT5','KIT','CDH1'],
    # TLS 
    'FRCs': ['PLA2G2A','PDPN','PDGFRA','TNFSF14'], #very few
    'TRCs' : ['PDGFRA','CCL19','ICAM1'], #,'CXCL12',
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    #'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    #'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'myCAFs': ['POSTN','MMP2','COL11A1'],
    'iCAFs' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10','CXCL12'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7','CCR7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    'CD4 Tfh' : ['CD4','ICOS','PDCD1','CXCL13'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD52','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','PRDM1'],
    # Myeloid
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    #'pDCs' :['GZMB'], #not present
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','TPSAB1'], 
    'LAM' : ['FPR3','APOE','APOC1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'MAC_SIGLEC1' : ['SELENOP','CSF1R','SIGLEC1','CD163']
}

gene_sets_by_region["TNBC_4"] = {
    'Malignant cells': ['EPCAM','KRT7','KRT8','NECTIN4','KRT19','MUC5B',],
    'Luminal progenitors' : ['KIT','CDH1','KRT19'],
    'Myoepithelial':['KRT5','ACTA2','KRT17'],
    # TLS 
    'FRCs': ['PLA2G2A','IGF1','PDGFRA','TNFSF14'], #very few
    #'FDCs': ['ICAM1','PDGFRA','CXCL13'], # check if exist
    'TRCs' : ['PDGFRA','CCL19','ICAM1'], #,'CXCL12',
    'LECs': ['LYVE1','FLT4','PROX1'], 
    # Stromal
    'Endothelial' : ['VWF','PLVAP','FLT1','KDR'],
    #'SMCs' : ['MYH11'], #don't use ACTA2
    'Pericytes': ['PDGFRB','RGS5'],
    #'Schwann': ['SOX10','CDH19','CHL1','NGFR'],
    #'Neurons': ['SYT1','CNTNAP2'],
    #'Adipocytes': ['PLIN1','LIPE','ADIPOQ'],
    # CAFs
    'myCAFs': ['POSTN','MMP2','FN1'],
    'iCAFs' : ['PDGFRA','SCN7A','C7','ABCA6','ABCA10'],
    # Lymphoid
    'NK': ['GNLY', 'NKG7','KLRC1'], #very few
    'CD4 T cells': ['IL7R','CD4','TCF7','CCR7'],
    'CD4 Treg' : ['FOXP3','IKZF2','CCR8'],
    #'CD4 Tfh' : ['CD4','ICOS','PDCD1','CXCL13'], 
    'CD8 T cells': ['CD8A','CD8B','GZMB','GZMA'],
    #'CD8 TRM' : ['CD8A','CD8B','ITGAE','ITGA1','STK17B','ID2'],
    'B cells': ['MS4A1','CD52','CD19'],
    #'B:GC' : {'LMO2','BCL6','CD22','MS4A1'},
    'Plasma': ['MZB1','PRDM1'],
    # Myeloid
    'DC1' : ['CLEC9A','IDO1'],
    'DC2' : ['CLEC10A','CD1C'],
    'pDCs' :['GZMB'],
    #'Langerhans DCs' : {'CDH20','CD207','CD1A'},
    'Migratory DCs' : ['CCL22','LAMP3'],
    'Mast' : ['CPA3','TPSAB1'], 
    #'LAM' : ['FPR3','APOE','APOC1'], #CSTB/SLA2A1 EXPRESSED BY MALIGNANT CELLS# 'SPP1' expressed by endothelial, ductal and TAMs, not present
    'MAC_SIGLEC1' : ['SELENOP','CSF1R','SIGLEC1','CD163']
}

# TLS_097_480_n2: the original labels were made with this earlier FDCs gene set (see the README);
# the notebook's FDCs set was changed afterwards without relabelling
gene_sets_by_region["TLS_097_480_n2"]["FDCs"] = ['VCAM1','FDCSP','CR1','CR2']

## 1. Crop, QC, label and neighbourhood enrichment for each region

Raw counts → crop → remove cells below `min_counts` and genes in fewer than 5 cells → normalise and log-transform, and keep a scaled layer for the matrix plots → score each gene set → label each cell with its highest score, or "Unclassified" if that score is below the region's threshold → neighbourhood enrichment on the labels.

In [4]:
def tls_region(name):
    settings = tls_regions[name]
    adata = sc.read_10x_h5(os.path.join(RAW_DIRS[name], "cell_feature_matrix.h5"))
    df = pd.read_csv(os.path.join(RAW_DIRS[name], "cells.csv"))
    df.set_index(adata.obs_names, inplace=True)
    adata.obs = df.copy()
    adata.obsm["spatial"] = adata.obs[["x_centroid", "y_centroid"]].copy().to_numpy()

    x_min, x_max, y_min, y_max = settings["box"]
    criteria = (
        (adata.obs['x_centroid'] >= x_min) & (adata.obs['x_centroid'] <= x_max) &
        (adata.obs['y_centroid'] >= y_min) & (adata.obs['y_centroid'] <= y_max)
    )
    subset_adata = adata[criteria, :].copy()

    sc.pp.filter_cells(subset_adata, min_counts=settings["min_counts"])
    sc.pp.filter_genes(subset_adata, min_cells=5)

    subset_adata.layers["counts"] = subset_adata.X.copy()
    sc.pp.normalize_total(subset_adata, inplace=True)
    sc.pp.log1p(subset_adata)
    subset_adata.layers['scaled'] = sc.pp.scale(subset_adata, copy=True).X

    gene_sets = gene_sets_by_region[name]
    for cell_type, genes in gene_sets.items():
        sc.tl.score_genes(subset_adata, gene_list=genes, score_name=f"{cell_type}_score")
    score_columns = [f"{ct}_score" for ct in gene_sets.keys() if f"{ct}_score" in subset_adata.obs.columns]
    score_data = subset_adata.obs[score_columns]

    subset_adata.obs["max_score"] = subset_adata.obs[score_columns].max(axis=1)
    subset_adata.obs["TLS_cell_class_threshold"] = subset_adata.obs[score_columns].idxmax(axis=1)
    subset_adata.obs["TLS_cell_class_threshold"] = subset_adata.obs["TLS_cell_class_threshold"].str.replace("_score", "")
    threshold = score_data.quantile(settings["quantile"])
    subset_adata.obs.loc[subset_adata.obs["max_score"] < threshold.max(), "TLS_cell_class_threshold"] = "Unclassified"
    if "unclassified_to" in settings:
        unclassified = subset_adata.obs["TLS_cell_class_threshold"] == "Unclassified"
        subset_adata.obs.loc[unclassified, "TLS_cell_class_threshold"] = settings["unclassified_to"]
    subset_adata.obs["TLS_cell_class_threshold"] = subset_adata.obs["TLS_cell_class_threshold"].astype("category")

    if settings["radius"] is None:
        sq.gr.spatial_neighbors(subset_adata, coord_type="generic", spatial_key="spatial")
    else:
        sq.gr.spatial_neighbors(subset_adata, coord_type="generic", spatial_key="spatial", radius=settings["radius"])
    sq.gr.nhood_enrichment(subset_adata, cluster_key="TLS_cell_class_threshold", seed=settings["seed"])
    print(name, subset_adata.n_obs, "cells")
    return subset_adata


regions = {}
for name in tls_regions:
    regions[name] = tls_region(name)
    regions[name].write_h5ad(os.path.join(OUTPUT_DIR, f"{name}.h5ad"))

  0%|          | 0/1000 [00:00<?, ?/s]

TLS_107_314 12573 cells


  0%|          | 0/1000 [00:00<?, ?/s]

TLS_104_314 11296 cells


  0%|          | 0/1000 [00:00<?, ?/s]

TLS_065_314 10280 cells


... (16 outputs left out)

  0%|          | 0/1000 [00:00<?, ?/s]

TNBC_3 46852 cells


  0%|          | 0/1000 [00:00<?, ?/s]

TNBC_4 38315 cells


## 2. Cells per label in each region

In [5]:
counts = pd.DataFrame({name: a.obs["TLS_cell_class_threshold"].value_counts() for name, a in regions.items()}).fillna(0).astype(int)
counts.loc["Total"] = counts.sum()
counts.to_csv(os.path.join(OUTPUT_DIR, "TLS_label_counts.tsv"), sep="\t")
counts

,TLS_107_314,TLS_104_314,TLS_065_314,TLS_096_480_n2,TLS_096_480_n1,TLS_097_480_n3,TLS_097_480_n2,TLS_097_480_n1,SLS_ref,TNBC_1,TNBC_2,TNBC_3,TNBC_4
TLS_cell_class_threshold,,,,,,,,,,,,,
Acinar_REG+,0,0,104,0,0,0,0,0,0,0,0,0,0
Adipocytes,0,0,0,25,61,0,0,0,210,0,0,0,0
B cells,213,1875,957,1142,2650,2010,2078,1918,9073,8377,7146,3693,3618
CAFs_CXCL14,755,1370,269,0,1414,620,585,0,0,0,0,0,0
CAFs_MYO,3097,660,365,660,1325,2104,360,1529,1191,0,0,0,0
CAFs_NRT,598,182,523,61,227,130,533,537,216,0,0,0,0
CD4 T cells,495,850,361,1240,2394,2022,3124,2608,5873,4702,1687,4492,2578
CD4 Tfh,0,0,19,162,282,126,66,82,272,749,0,1460,0
CD4 Treg,66,70,48,313,465,373,201,122,454,935,194,802,129


## 3. Check against the original region objects

In a passing run, the cells are the same, 100% of them have the same label, and the neighbourhood enrichment z-scores match (largest absolute difference 0). 

In [6]:
rows = []
for name, path in ARCHIVED_FILES.items():
    if not path or not os.path.exists(path):
        print(f"{name}: no original region object")
        continue
    reference = sc.read_h5ad(path, backed="r")
    new = regions[name]
    shared = new.obs_names.intersection(reference.obs_names)
    row = {
        "region": name,
        "cells": new.n_obs,
        "cells_original": reference.n_obs,
        "shared": len(shared),
        "same_label": f"{(new.obs.loc[shared, 'TLS_cell_class_threshold'].astype(str) == reference.obs.loc[shared, 'TLS_cell_class_threshold'].astype(str).replace(RENAMED_LABELS)).mean():.4%}",
        "max_zscore_difference": None,
    }
    key = "TLS_cell_class_threshold_nhood_enrichment"
    if key in reference.uns:
        # Compare by label name, because the renamed labels sort differently
        old_labels = [RENAMED_LABELS.get(l, l) for l in reference.obs["TLS_cell_class_threshold"].cat.categories]
        new_labels = list(new.obs["TLS_cell_class_threshold"].cat.categories)
        if sorted(old_labels) == sorted(new_labels):
            old_z = pd.DataFrame(reference.uns[key]["zscore"], index=old_labels, columns=old_labels).loc[new_labels, new_labels]
            row["max_zscore_difference"] = float(np.nanmax(np.abs(new.uns[key]["zscore"] - old_z.to_numpy())))
        else:
            row["max_zscore_difference"] = "different labels"
    rows.append(row)
    reference.file.close()
check = pd.DataFrame(rows)
check.to_csv(os.path.join(OUTPUT_DIR, "stage3_TLS_check.tsv"), sep="\t", index=False)
display(check)

,region,cells,cells_original,shared,same_label,max_zscore_difference
0,TLS_107_314,12573,12573,12573,100.0000%,0.0
1,TLS_104_314,11296,11296,11296,100.0000%,0.0
2,TLS_065_314,10280,10280,10280,100.0000%,0.0
3,TLS_096_480_n2,14070,14070,14070,100.0000%,0.0
4,TLS_096_480_n1,26559,26559,26559,100.0000%,0.0
5,TLS_097_480_n3,19076,19076,19076,100.0000%,0.0
6,TLS_097_480_n2,18841,18841,18841,100.0000%,0.0
7,TLS_097_480_n1,17815,17815,17815,100.0000%,0.0
8,SLS_ref,36076,36076,36076,100.0000%,0.0
9,TNBC_1,54017,54017,54017,100.0000%,0.0
